In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())


2.5.1
11.8
True


In [2]:
import labelme2coco


labelme_train = "/home/admin/Documents/AI_Eng/Week_05/Data/Task_02/train/"
json_train = "/home/admin/Documents/AI_Eng/Week_05/Data/Task_02/train_coco"

labelme_test = "/home/admin/Documents/AI_Eng/Week_05/Data/Task_02/test/"
json_test = "/home/admin/Documents/AI_Eng/Week_05/Data/Task_02/test_coco"


def convert_json_to_coco(json_path, export_path):
    """
    Convert JSON annotations to COCO format.
    
    Args:
        json_path (str): Path to the input JSON file.
        export_path (str): Path to save the output COCO JSON file.
    """
    # Convert JSON to COCO format
    labelme2coco.convert(json_path, export_path)
    print(f"Converted {json_path} to COCO format and saved to {export_path}")

# Convert training annotations
convert_json_to_coco(labelme_train, json_train)
# Convert testing annotations
convert_json_to_coco(labelme_test, json_test)

There are 590 listed files in folder .


Converting labelme annotations to COCO format: 100%|██████████| 590/590 [00:01<00:00, 356.48it/s]
04/14/2025 14:27:47 - INFO - labelme2coco -   Converted annotations in COCO format is exported to /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/train_coco/dataset.json


Converted /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/train/ to COCO format and saved to /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/train_coco
There are 10 listed files in folder .


Converting labelme annotations to COCO format: 100%|██████████| 10/10 [00:00<00:00, 347.77it/s]
04/14/2025 14:27:47 - INFO - labelme2coco -   Converted annotations in COCO format is exported to /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/test_coco/dataset.json


Converted /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/test/ to COCO format and saved to /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/test_coco


In [3]:
import os
import torch
import torchvision
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import json
import torchvision.transforms as T

##################################
# SECTION 1: Data Loading
##################################
print("SECTION 1: Data Loading...")


train_json = os.path.join(json_train, "dataset.json")
train_images = labelme_train

test_json = os.path.join(json_test, "dataset.json")
test_images = labelme_test

print(f"Train JSON: {train_json}")
print(f"Train Images Dir: {train_images}")
print(f"Test JSON: {test_json}")
print(f"Test Images Dir: {test_images}")

# Custom Dataset for COCO Annotations
class LogsDataset(Dataset):
    def __init__(self, coco_json, images_dir, transforms=None):
        self.images_dir = images_dir
        self.transforms = transforms
        # Load COCO-like JSON
        with open(coco_json, 'r') as f:
            data = json.load(f)
        self.images = data['images']
        self.annotations = data['annotations']
        self.categories = data['categories']

        # Build index of image_id -> list of annotation indexes
        self.img_id_to_ann = {}
        for i, ann in enumerate(self.annotations):
            img_id = ann['image_id']
            if img_id not in self.img_id_to_ann:
                self.img_id_to_ann[img_id] = []
            self.img_id_to_ann[img_id].append(i)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        file_name = img_info['file_name']
        img_path = os.path.join(self.images_dir, file_name)
        
        # Load image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB
        
        # Gather annotations
        ann_ids = self.img_id_to_ann.get(img_id, [])
        boxes = []
        labels = []
        
        for ann_id in ann_ids:
            ann = self.annotations[ann_id]
            # Single class "log" => label=1
            labels.append(1)
            # Bbox in COCO is [x, y, width, height]
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x + w, y + h])

        # Convert to torch tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        masks = torch.zeros((len(boxes), img.shape[0], img.shape[1]), dtype=torch.uint8)

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target



SECTION 1: Data Loading...
Train JSON: /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/train_coco/dataset.json
Train Images Dir: /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/train/
Test JSON: /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/test_coco/dataset.json
Test Images Dir: /home/admin/Documents/AI_Eng/Week_05/Data/Task_02/test/


In [4]:

# Transforms
transform = T.Compose([
    T.ToTensor(),
])

# Create datasets
dataset_train = LogsDataset(train_json, train_images, transforms=transform)
dataset_test  = LogsDataset(test_json, test_images, transforms=transform)

print(f"Number of training images: {len(dataset_train)}")
print(f"Number of testing images: {len(dataset_test)}")

# Create dataloaders
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    dataset_train,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    dataset_test,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print("Data Loading Complete.\n")

Number of training images: 590
Number of testing images: 10
Data Loading Complete.



In [5]:
##################################
# SECTION 2: Model Setup
##################################
print("SECTION 2: Model Setup...")

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Device being used: {device}")

# Load pre-trained PyTorch Mask R-CNN
model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
print("Loaded pre-trained Mask R-CNN.")

# Replace classifier head (box predictor)
num_classes = 2  # 1 (log) + background
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features, num_classes
)

# Replace mask predictor
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
    in_features_mask, hidden_layer, num_classes
)

model.to(device)
print("Model moved to device.")
print("Model Setup Complete.\n")


SECTION 2: Model Setup...
Device being used: cuda


/home/admin/anaconda3/envs/AI_Engineering/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/admin/anaconda3/envs/AI_Engineering/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loaded pre-trained Mask R-CNN.
Model moved to device.
Model Setup Complete.



In [6]:
##################################
# SECTION 3: Training & Testing
##################################
print("SECTION 3: Training & Testing...")

# Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10
print(f"Starting training for {num_epochs} epochs...")

model.train()
for epoch in range(num_epochs):
    print(f"Starting Epoch {epoch+1}...")
    total_loss = 0.0
    
    for step, (imgs, targets) in enumerate(train_loader):
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        if (step + 1) % 10 == 0:
            print(f"  [Step {step+1}] Loss: {losses.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")

print("Training completed.\n")


SECTION 3: Training & Testing...
Starting training for 10 epochs...
Starting Epoch 1...
  [Step 10] Loss: 0.5356
  [Step 20] Loss: 0.3231
  [Step 30] Loss: 0.2404
  [Step 40] Loss: 0.2434
  [Step 50] Loss: 0.1850
  [Step 60] Loss: 0.1873
  [Step 70] Loss: 0.1783
  [Step 80] Loss: 0.1363
  [Step 90] Loss: 0.1370
  [Step 100] Loss: 0.1518
  [Step 110] Loss: 0.1797
  [Step 120] Loss: 0.1457
  [Step 130] Loss: 0.1868
  [Step 140] Loss: 0.1350
  [Step 150] Loss: 0.1389
  [Step 160] Loss: 0.1469
  [Step 170] Loss: 0.1252
  [Step 180] Loss: 0.1415
  [Step 190] Loss: 0.1427
  [Step 200] Loss: 0.1253
  [Step 210] Loss: 0.1739
  [Step 220] Loss: 0.1529
  [Step 230] Loss: 0.1569
  [Step 240] Loss: 0.1540
  [Step 250] Loss: 0.1290
  [Step 260] Loss: 0.1567
  [Step 270] Loss: 0.1138
  [Step 280] Loss: 0.1423
  [Step 290] Loss: 0.1455
Epoch [1/10] - Average Loss: 0.1914
Starting Epoch 2...
  [Step 10] Loss: 0.1177
  [Step 20] Loss: 0.1042
  [Step 30] Loss: 0.1010
  [Step 40] Loss: 0.1246
  [Step 50]

In [7]:
print("Starting Evaluation / Testing")
model.eval()
log_counts = {}

with torch.no_grad():
    for i, (imgs, targets) in enumerate(test_loader):
        imgs = [img.to(device) for img in imgs]
        outputs = model(imgs)

        for img_i, output in enumerate(outputs):
            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            threshold = 0.5
            keep = scores >= threshold
            count_logs = sum(keep)
            log_counts[f"batch{i}_img{img_i}"] = count_logs

print("Testing completed.")
print("Log Counts:", log_counts)

print("\nSECTION 3: Training & Testing Complete.")

Starting Evaluation / Testing...
Testing completed.
Log Counts: {'batch0_img0': np.int64(13), 'batch0_img1': np.int64(13), 'batch1_img0': np.int64(11), 'batch1_img1': np.int64(11), 'batch2_img0': np.int64(12), 'batch2_img1': np.int64(12), 'batch3_img0': np.int64(11), 'batch3_img1': np.int64(11), 'batch4_img0': np.int64(11), 'batch4_img1': np.int64(11)}

SECTION 3: Training & Testing Complete.


In [10]:
torch.save(model, 'maskrcnn_model_full.pth')
print("Entire model saved as 'maskrcnn_model_full.pth'")

Entire model saved as 'maskrcnn_model_full.pth'


In [16]:
import cv2
import numpy as np

def draw_predictions(image, boxes, scores, threshold=0.5):
    
    image = np.ascontiguousarray(image)
    for box, score in zip(boxes, scores):
        if score >= threshold:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, f"log: {score:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return image


with torch.no_grad():
    for i, (imgs, targets) in enumerate(test_loader):
        imgs = list(img.to(device) for img in imgs)
        outputs = model(imgs)
        
        for img_i, output in enumerate(outputs):
            # Convert tensor image back to NumPy (after moving to CPU)
            np_img = imgs[img_i].permute(1, 2, 0).cpu().numpy()
            np_img = (np_img * 255).astype(np.uint8)
            
            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            
            
            # Draw predictions and capture the modified image
            np_img = draw_predictions(np_img, boxes, scores, threshold=0.5)
            
            # Save result; converting color from RGB to BGR for OpenCV
            cv2.imwrite(f"Data/Task_02/LogOutput/detected_batch{i}_img{img_i}.jpg", cv2.cvtColor(np_img, cv2.COLOR_RGB2BGR))
